# WildPulse image review

Upload JPEG or PNG images, run wildlife detection, then download the results. Runs on CPU. Model startup and larger batches can take several minutes. Images stay in your Colab session; this notebook does not publish them to GitHub. Colab storage is temporary.

Predictions cover animals, people and vehicles, not species. The default confidence of 0.5 is for review and is not a production recommendation.

## Set up
Run this cell once per Colab session.

In [ ]:
%pip -q install sparrow-engine==0.1.32 opencv-python-headless
from pathlib import Path
from urllib.request import urlopen
import importlib.util
script = Path("/content/run_images.py")
script.write_bytes(urlopen("https://raw.githubusercontent.com/Shreyakb301/WildPulse/0d33fd1e4ee26785a45f91eebc61a12771c230f9/run_images.py", timeout=60).read())
spec = importlib.util.spec_from_file_location("wildpulse_images", script)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
print("WildPulse is ready")


## Upload images
Each upload starts a separate batch. Choose JPEG or PNG files from your computer.

In [ ]:
from google.colab import files
import tempfile
summary, bundle = None, None
images = None
uploaded = files.upload()
if not uploaded:
    raise ValueError('No images selected. Upload images or use the example below.')
if any(Path(name).suffix.lower() not in {'.jpg', '.jpeg', '.png'} for name in uploaded):
    raise ValueError('Please choose JPEG or PNG images only.')
batch = Path(tempfile.mkdtemp(prefix='wildpulse_', dir='/content'))
images = batch/'images'
images.mkdir()
for name, content in uploaded.items():
    (images/Path(name).name).write_bytes(content)
print(f'Uploaded {len(uploaded)} images')


### Or try an example
Skip the upload cell and run this cell to try one public [Snapshot Enonkishu](https://lila.science/datasets/snapshot-enonkishu) camera image. It starts a separate batch. This frame is strongly overexposed and demonstrates the need for visual review. Species is not identified.

In [ ]:
# Public Snapshot Enonkishu example. This is an animal-presence demonstration.
import tempfile, hashlib
summary, bundle = None, None
images = None
batch = Path(tempfile.mkdtemp(prefix='wildpulse_example_', dir='/content'))
example_dir = batch/'images'
example_dir.mkdir()
url = 'https://lilawildlife.blob.core.windows.net/lila-wildlife/snapshot-safari/ENO/ENO_public/ENO_S1/B06/B06_R2/ENO_S1_B06_R2_IMAG0434.JPG'
raw = urlopen(url, timeout=60).read()
if hashlib.sha256(raw).hexdigest() != '83924ec1daa03e20eb17ef82d5591368eeec992772c3c4faa0e7968e7e7c0494':
    raise ValueError('Example changed. Stop and review its source before continuing.')
(example_dir/'example.JPG').write_bytes(raw)
images = example_dir
print('Example loaded. Run Detect and preview next.')


## Detect and preview
Run after uploading images or loading the example. You can change confidence and run again; each run saves separate results. Unreadable images are listed in the report. Unexpected detection errors stop the run.

In [ ]:
from IPython.display import display, Image
confidence = 0.5 # @param {type:"slider", min:0.1, max:1.0, step:0.1}
summary, bundle = None, None
if images is None or not images.is_dir():
    raise ValueError('Upload images or load the example first.')
import tempfile
run_parent = Path(tempfile.mkdtemp(prefix='run_', dir=str(batch)))
summary, bundle = workflow.run(images, run_parent/'results', threshold=confidence)
print(f"Processed {summary['successful']} images; {summary['failed']} unreadable")
for record in summary["records"]:
    if record["status"] == "ok":
        print(record["image"])
        display(Image(filename=str(bundle.parent/record["preview"]), width=800))


## Download results
The ZIP contains annotated previews, JSON results and CSV detections. Download before the session ends. Original uploads are not included in the ZIP.

In [ ]:
if bundle is None or not bundle.is_file():
    raise ValueError('Complete a detection run before downloading.')
from google.colab import files
files.download(str(bundle))


### If the download does not start
Run this cell for a direct download link for bundles up to 20 MB. Larger bundles can be downloaded from the Colab Files pane.

In [ ]:
import base64, hashlib, zipfile
from IPython.display import HTML, display
if bundle is None or not bundle.is_file():
    raise ValueError('Complete a detection run before downloading.')
raw = bundle.read_bytes()
with zipfile.ZipFile(bundle) as archive:
    assert archive.testzip() is None
print('ZIP bytes:', len(raw), 'SHA256:', hashlib.sha256(raw).hexdigest())
if len(raw) <= 20_000_000:
    display(HTML('<a download="WildPulse_results.zip" href="data:application/zip;base64,'+base64.b64encode(raw).decode()+'">Download WildPulse ZIP</a>'))
else:
    print('Large bundle: use the standard download cell or Colab Files pane')
